# Networking RAG - Phase 4 Evaluation

This notebook evaluates the improved Networking RAG system developed in Phase 4. The evaluation is performed using the same test questions and RAGAS metrics used previously, allowing comparison between the baseline and improved retrieval pipelines.

In [ ]:
# Install required libraries

try:

    !pip install -r requirements.txt

    print(
        "Dependencies installed successfully!"
    )

except Exception as e:

    print(
        "Installation Error:"
    )

    print(e)

In [ ]:
# Import required libraries

import os
import time
import pickle
import chromadb
import pandas as pd

from datasets import Dataset

from google import genai
from google.colab import userdata

from groq import Groq

print(
    "Libraries imported successfully!"
)

In [ ]:
# Import RAGAS

try:

    import ragas

    print(
        "RAGAS imported successfully!"
    )

    print(
        "Version:",
        ragas.__version__
    )

except Exception as e:

    print(e)

In [ ]:
# Initialize Gemini

try:

    gemini_client = genai.Client(
        api_key=userdata.get(
            "GEMINI_API_KEY"
        )
    )

    print(
        "Gemini initialized!"
    )

except Exception as e:

    print(e)

In [ ]:
# Initialize Groq

try:

    groq_client = Groq(
        api_key=userdata.get(
            "GROQ_API_KEY"
        )
    )

    print(
        "Groq initialized!"
    )

except Exception as e:

    print(e)

In [ ]:
# Load ChromaDB

try:

    chroma_client = chromadb.PersistentClient(
        path="./networking_chromadb_phase4"
    )

    collection = (
        chroma_client.get_or_create_collection(
            name="networking_docs_phase4"
        )
    )

    print(
        "Records:",
        collection.count()
    )

except Exception as e:

    print(e)

In [ ]:
# Load saved chunks and embeddings

try:

    with open(
        "improved_chunks.pkl",
        "rb"
    ) as f:

        improved_chunks = pickle.load(f)

    with open(
        "improved_embeddings.pkl",
        "rb"
    ) as f:

        improved_embeddings = pickle.load(f)

    if collection.count() == 0:

        collection.add(

            ids=[
                f"chunk_{i}"
                for i in range(
                    len(improved_chunks)
                )
            ],

            documents=
            improved_chunks,

            embeddings=
            improved_embeddings
        )

    print(
        "Collection ready!"
    )

    print(
        "Records:",
        collection.count()
    )

except Exception as e:

    print(e)

In [ ]:
# Load evaluation questions

try:

    def load_evaluation_data(
        path="arrays.txt"
    ):

        globals_dict = {}

        with open(
            path,
            "r"
        ) as f:

            exec(
                f.read(),
                globals_dict
            )

        return (

            globals_dict[
                "test_questions"
            ],

            globals_dict[
                "ground_truths"
            ]

        )

    test_questions, ground_truths = (

        load_evaluation_data()

    )

    # Keep only first 5 questions

    test_questions = (
        test_questions[:5]
    )

    ground_truths = (
        ground_truths[:5]
    )

    print(
        "Questions:",
        len(test_questions)
    )

    print(
        "Ground Truths:",
        len(ground_truths)
    )

except Exception as e:

    print(
        "Loading Error:"
    )

    print(e)

In [ ]:
# Install BM25 library

try:

    !pip install -q rank-bm25

    print(
        "BM25 library installed successfully!"
    )

except Exception as e:

    print(
        "Installation Error:"
    )

    print(e)

In [ ]:
# Create BM25 index

try:

    from rank_bm25 import BM25Okapi

    tokenized_chunks = [

        chunk.lower().split()

        for chunk in improved_chunks

    ]

    bm25 = BM25Okapi(
        tokenized_chunks
    )

    print(
        "BM25 index created successfully!"
    )

    print(
        "Indexed Chunks:",
        len(tokenized_chunks)
    )

except Exception as e:

    print(e)

In [ ]:
# RAG Query Function

try:

    def ask_network_question(
        question
    ):

        query_response = (
            gemini_client.models.embed_content(
                model="models/gemini-embedding-001",
                contents=question
            )
        )

        query_embedding = (
            query_response.embeddings[0].values
        )

        vector_results = (
            collection.query(
                query_embeddings=[
                    query_embedding
                ],
                n_results=5
            )
        )

        vector_chunks = (
            vector_results["documents"][0]
        )

        tokenized_query = (
            question.lower().split()
        )

        bm25_scores = (
            bm25.get_scores(
                tokenized_query
            )
        )

        bm25_indices = (
            sorted(
                range(
                    len(bm25_scores)
                ),
                key=lambda i:
                bm25_scores[i],
                reverse=True
            )[:5]
        )

        bm25_chunks = [

            improved_chunks[i]

            for i in bm25_indices

        ]

        retrieved_chunks = []

        for chunk in (
            vector_chunks +
            bm25_chunks
        ):

            if chunk not in retrieved_chunks:

                retrieved_chunks.append(
                    chunk
                )

        retrieved_chunks = (
            retrieved_chunks[:5]
        )

        context = "\n\n".join(
            retrieved_chunks
        )

        prompt = f"""
You are a networking assistant.

Answer only using the provided context.

Context:
{context}

Question:
{question}
"""

        response = (
            groq_client.chat.completions.create(
                model=
                "llama-3.3-70b-versatile",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )
        )

        return {

            "answer":
            response.choices[0]
            .message.content,

            "contexts":
            retrieved_chunks
        }

    print(
        "Function created successfully!"
    )

except Exception as e:

    print(e)

In [ ]:
# Test hybrid search

try:

    result = (
        ask_network_question(
            "What is DNS?"
        )
    )

    print(
        result["answer"]
    )

except Exception as e:

    print(e)

In [ ]:
# Generate answers for evaluation questions

try:

    generated_answers = []

    retrieved_contexts = []

    for i, question in enumerate(
        test_questions
    ):

        print(
            f"Processing {i+1}/{len(test_questions)}"
        )

        result = (
            ask_network_question(
                question
            )
        )

        generated_answers.append(
            result["answer"]
        )

        retrieved_contexts.append(
            result["contexts"]
        )

        if (
            (i + 1) % 2 == 0
            and
            (i + 1) < len(
                test_questions
            )
        ):

            print(
                "Rate limit protection activated. Waiting 30 seconds..."
            )

            time.sleep(30)

    print(
        "Generation completed!"
    )

except Exception as e:

    print(
        "Generation Error:"
    )

    print(e)

In [ ]:
# Create Evaluation Dataset

try:

    evaluation_dataset = (
        Dataset.from_dict(
            {
                "question":
                test_questions,

                "answer":
                generated_answers,

                "contexts":
                retrieved_contexts,

                "ground_truth":
                ground_truths
            }
        )
    )

    print(
        evaluation_dataset
    )

except Exception as e:

    print(
        "Dataset Error:"
    )

    print(e)

In [ ]:
# Configure evaluator models

try:

    from langchain_groq import ChatGroq

    from langchain_google_genai import (
        GoogleGenerativeAIEmbeddings
    )

    evaluator_llm = (
        ChatGroq(
            model=
            "llama-3.3-70b-versatile",

            api_key=
            userdata.get(
                "GROQ_API_KEY"
            )
        )
    )

    evaluator_embeddings = (
        GoogleGenerativeAIEmbeddings(
            model=
            "models/gemini-embedding-001",

            google_api_key=
            userdata.get(
                "GEMINI_API_KEY"
            )
        )
    )

    print(
        "Evaluator configured successfully!"
    )

except Exception as e:

    print(
        "Configuration Error:"
    )

    print(e)

In [ ]:
# Import RAGAS metrics

try:

    from ragas import evaluate

    from ragas.metrics import (
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    )

    print(
        "Metrics imported successfully!"
    )

except Exception as e:

    print(
        "Import Error:"
    )

    print(e)

In [ ]:
from ragas.run_config import RunConfig

run_config = RunConfig(
    timeout=300
)

In [ ]:
# Evaluate improved RAG pipeline

try:

    results = evaluate(

        evaluation_dataset,

        metrics=[

            faithfulness,

            answer_relevancy,

            context_precision,

            context_recall

        ],

        llm=evaluator_llm,

        embeddings=
        evaluator_embeddings,
        run_config=run_config
    )

    print(
        "Evaluation completed successfully!"
    )

    print(
        results
    )

except Exception as e:

    print(
        "Evaluation Error:"
    )

    print(e)

In [ ]:
# Save evaluation results

try:

    results_df = (
        results.to_pandas()
    )

    print(
        results_df
    )

    results_df.to_csv(

        "phase4_ragas_results.csv",

        index=False

    )

    print(
        "Results saved successfully!"
    )

except Exception as e:

    print(e)